# Case study: leaderboard round L2 with gradient boosting

*Session 10, block 3 practice. Author: course team, licence CC-BY-4.0.*

We build the leakage-free feature table of Session 9, tune a LightGBM model with a **time-based** cross-validation and macro-F1, compare it with the current model (a logistic regression on simple features, round L1), inspect it with permutation importance and SHAP, and write the submission file.

Rules from Session 9:

- **training rows:** product rating = mean of *earlier* reviews of the same product (`prod_prev_mean`), never `train_avg_rating`;
- **test rows:** `train_avg_rating` and `train_n_reviews` from `products.parquet` are the correct as-of values, because all training reviews are older than the test reviews.

By default the model is trained on the 50,000-review sample; set `FULL = True` to use all 434,373 training reviews (slower, usually a little better).

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "case-study" / "data").exists())
DATA = ROOT / "case-study" / "data"
FULL = False

train = pd.read_parquet(DATA / "train.parquet").sort_values(["date", "review_id"]).reset_index(drop=True)
test = pd.read_parquet(DATA / "test.parquet")
products = pd.read_parquet(DATA / "products.parquet", columns=["parent_asin", "store", "train_avg_rating", "train_n_reviews"])
print(len(train), len(test))

## 1. Past-only history features

In [ ]:
g = train.groupby("parent_asin")["rating"]
train["prod_prev_n"] = g.cumcount()
train["prod_prev_mean"] = ((g.cumsum() - train["rating"]) / train["prod_prev_n"]).where(train["prod_prev_n"] > 0)
train["user_prev_n"] = train.groupby("user_id").cumcount()

# the same quantities for the test rows, from the training period only
user_counts = train["user_id"].value_counts()
test = test.merge(products[["parent_asin", "train_avg_rating", "train_n_reviews"]], on="parent_asin", how="left")
test["prod_prev_mean"] = test["train_avg_rating"]
test["prod_prev_n"] = test["train_n_reviews"].fillna(0)
test["user_prev_n"] = test["user_id"].map(user_counts).fillna(0)

if FULL:
    fit_df = train
else:
    sample_ids = pd.read_parquet(DATA / "train_sample.parquet", columns=["review_id"])["review_id"]
    fit_df = train[train["review_id"].isin(sample_ids)]
fit_df = fit_df.merge(products[["parent_asin", "store"]], on="parent_asin", how="left").reset_index(drop=True)
test = test.merge(products[["parent_asin", "store"]], on="parent_asin", how="left")
print(len(fit_df), fit_df["date"].is_monotonic_increasing)

## 2. The feature table

In [ ]:
NEGATIONS = r"\b(?:not|no|never|don't|doesn't|didn't|isn't|wasn't|won't|can't)\b"


def features(frame):
    """Leakage-free features of Session 9 for training or test rows."""
    text, title = frame["text"].fillna(""), frame["title"].fillna("")
    n_chars = text.str.len()
    month = frame["date"].dt.month
    return pd.DataFrame({
        "log_chars": np.log1p(n_chars),
        "n_words": text.str.split().str.len(),
        "n_exclaim": text.str.count("!"),
        "n_question": text.str.count(r"\?"),
        "upper_share": text.str.count(r"[A-Z]") / n_chars.clip(lower=1),
        "n_negations": text.str.lower().str.count(NEGATIONS),
        "title_words": title.str.split().str.len(),
        "title_exclaim": title.str.count("!"),
        "verified": frame["verified_purchase"].astype(int),
        "helpful_vote": frame["helpful_vote"],
        "n_images": frame["n_images"],
        "year": frame["date"].dt.year,
        "month_sin": np.sin(2 * np.pi * month / 12),
        "month_cos": np.cos(2 * np.pi * month / 12),
        "prod_prev_mean": frame["prod_prev_mean"],
        "prod_prev_n": frame["prod_prev_n"],
        "user_prev_n": frame["user_prev_n"],
        "store": frame["store"].fillna("missing"),
    }, index=frame.index)


X, y = features(fit_df), fit_df["label"]
X_test = features(test)
print(X.shape, X_test.shape)
X.describe().T.round(2)

**Task 1.** Compare the distributions of `prod_prev_n` and `prod_prev_mean` in `X` and `X_test`. Why are the test products' counts larger on average? Is that a problem for the model?

In [ ]:
# TODO (Task 1)


## 3. Current model and challenger on the same time-based folds

The **current model** is the L1 approach: logistic regression on simple text features with class weights. The **challenger** is LightGBM with balanced class weights; the store is target-encoded with cross-fitting inside the pipeline.

In [ ]:
from lightgbm import LGBMClassifier
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import KFold, TimeSeriesSplit, cross_val_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler, TargetEncoder

cv = TimeSeriesSplit(n_splits=5)
simple = ["log_chars", "n_exclaim", "n_question", "n_negations", "verified", "helpful_vote", "n_images"]
numeric = [c for c in X.columns if c != "store"]

current = make_pipeline(
    ColumnTransformer([("simple", StandardScaler(), simple)]),
    LogisticRegression(max_iter=2000, class_weight="balanced"))

challenger = make_pipeline(
    ColumnTransformer([("num", "passthrough", numeric),
                       ("store", TargetEncoder(target_type="multiclass", cv=KFold(5, shuffle=True, random_state=0)),
                        ["store"])]),
    LGBMClassifier(n_estimators=400, learning_rate=0.05, num_leaves=31, min_child_samples=50, subsample=0.8,
                   subsample_freq=1, colsample_bytree=0.8, class_weight="balanced", n_jobs=1, verbose=-1,
                   random_state=0))

s_cur = cross_val_score(current, X, y, cv=cv, scoring="f1_macro")
s_ch = cross_val_score(challenger, X, y, cv=cv, scoring="f1_macro")
pd.DataFrame({"current": s_cur, "challenger": s_ch, "difference": s_ch - s_cur}).round(3)

## 4. Tune the challenger

A small randomised search with the time-based folds. `n_jobs=1` in LightGBM keeps the two levels of parallelism from fighting.

In [ ]:
import time

from scipy.stats import loguniform, randint, uniform
from sklearn.model_selection import RandomizedSearchCV

t0 = time.time()
search = RandomizedSearchCV(
    challenger,
    {"lgbmclassifier__learning_rate": loguniform(0.02, 0.2),
     "lgbmclassifier__num_leaves": randint(8, 128),
     "lgbmclassifier__min_child_samples": randint(10, 200),
     "lgbmclassifier__colsample_bytree": uniform(0.5, 0.5)},
    n_iter=8, scoring="f1_macro", cv=cv, random_state=0).fit(X, y)
print({k.split("__")[1]: round(float(v), 3) for k, v in search.best_params_.items()})
print("best time-based CV macro-F1:", round(search.best_score_, 3), "|", round(time.time() - t0), "s")

**Task 2.** Increase `n_iter` or add `n_estimators` to the search. Does the CV score still improve? Write down the CV score next to the public leaderboard score after you submit. Expect the leaderboard score to be lower than the time-based CV score: the test years 2022–2023 have more negative reviews than the training years (drift, Session 16).

**Task 2b.** Drop `year` from the features and compare. A tree model treats every test review like a 2021 review; can a feature that helps in cross-validation hurt on future data?

## 5. Check the model: permutation importance and SHAP

We fit the tuned model on all but the last fold and inspect it on the last (most recent) fold.

In [ ]:
from sklearn.inspection import permutation_importance
from sklearn.metrics import classification_report

tr_idx, va_idx = list(cv.split(X))[-1]
best = search.best_estimator_.fit(X.iloc[tr_idx], y.iloc[tr_idx])
print(classification_report(y.iloc[va_idx], best.predict(X.iloc[va_idx]), digits=3))
perm = permutation_importance(best, X.iloc[va_idx], y.iloc[va_idx], scoring="f1_macro", n_repeats=5, random_state=0)
pd.Series(perm.importances_mean, index=X.columns).sort_values(ascending=False).round(4)

In [ ]:
import shap

prep, lgbm = best[0], best[-1]
X_va_enc = pd.DataFrame(prep.transform(X.iloc[va_idx]), columns=prep.get_feature_names_out())
sv = shap.TreeExplainer(lgbm)(X_va_enc.iloc[:2000])
print(sv.values.shape)                       # rows x features x classes (neg, neu, pos)
mean_abs = pd.DataFrame(np.abs(sv.values).mean(axis=0), index=X_va_enc.columns, columns=lgbm.classes_)
mean_abs.sort_values("neu", ascending=False).round(3).head(10)

**Task 3.** Is any feature suspiciously dominant? Replace `prod_prev_mean` by the leaky `train_avg_rating` for the training rows, refit, and compare the SHAP summary and the CV score. What would happen on the leaderboard?

## 6. Fit on all training rows and write the submission

In [ ]:
final = search.best_estimator_.fit(X, y)
submission = pd.DataFrame({"review_id": test["review_id"], "label": final.predict(X_test)})
print(submission["label"].value_counts(normalize=True).round(3))
submission.to_csv("submission_l2.csv", index=False)
print("written:", Path("submission_l2.csv").resolve())

Upload `submission_l2.csv` to the leaderboard (or give it to the lecturer). Add to your team repository: the time-based CV score next to the public leaderboard score, the comparison with the current model, and the permutation importance of the final model.